# HabitLab Fine-tuning — Clean Version
Fine-tunes Qwen2.5-1.5B-Instruct on Kaggle free GPU (T4).
No bitsandbytes. Single GPU. fp16. Pinned package versions to avoid conflicts.

**Steps:**
1. Run Cell 1 (installs) → **Restart kernel** → Run all remaining cells
2. Training takes ~15–20 min
3. Download `habitlab_model.gguf` from the Output tab

In [ ]:
# CELL 1 — Install packages with pinned versions
# After this cell finishes: Kernel → Restart, then run all cells below
%%capture
!pip install transformers==4.44.0 peft==0.12.0 accelerate==0.33.0 datasets -q

In [ ]:
# CELL 2 — Force single GPU (must run before anything else)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# CELL 3 — Load base model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda:0'
)
model.config.use_cache = False
model.enable_input_require_grads()

print('Model loaded. VRAM used:', round(torch.cuda.memory_allocated(0)/1e9, 1), 'GB')

In [ ]:
# CELL 4 — Attach LoRA adapters
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.0,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# CELL 5 — Load training data
import json
from datasets import Dataset

# Auto-find the uploaded combined_training_data.jsonl
DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'combined_training_data' in f:
            DATA_PATH = os.path.join(root, f)
            break

if DATA_PATH is None:
    raise FileNotFoundError('combined_training_data.jsonl not found. Make sure you added it as a dataset input.')

print('Data found at:', DATA_PATH)

raw_data = []
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))
print(f'Loaded {len(raw_data)} examples.')

def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_example)
print('Dataset ready.')

In [ ]:
# CELL 6 — Train
import os
os.environ['WANDB_DISABLED'] = 'true'  # Skip W&B prompt

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def tokenize_fn(examples):
    out = tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )
    out['labels'] = out['input_ids'].copy()
    return out

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

training_args = TrainingArguments(
    output_dir='/kaggle/working/outputs',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim='adamw_torch',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    dataloader_pin_memory=False,
    report_to='none',  # Disable all logging integrations
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Training started...')
result = trainer.train()
print(f'Done. Time: {round(result.metrics["train_runtime"]/60, 1)} minutes')

In [ ]:
# CELL 7 — Save adapter and merge into base model
from peft import PeftModel

print('Saving adapter...')
model.save_pretrained('/kaggle/working/lora_adapter')
tokenizer.save_pretrained('/kaggle/working/lora_adapter')

print('Merging into base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu'
)
merged = PeftModel.from_pretrained(base_model, '/kaggle/working/lora_adapter')
merged = merged.merge_and_unload()
merged.save_pretrained('/kaggle/working/merged_model')
tokenizer.save_pretrained('/kaggle/working/merged_model')
print('Merged model saved.')

In [ ]:
# CELL 8 — Convert to GGUF for Ollama
import os

# Clone llama.cpp (remove old copy if it exists)
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp --depth 1 -q
!pip install -r llama.cpp/requirements.txt -q

# Convert — q8_0 is the supported quantization in current llama.cpp
!python llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged_model \
    --outtype q8_0 \
    --outfile /kaggle/working/habitlab_model.gguf

if os.path.exists('/kaggle/working/habitlab_model.gguf'):
    size = os.path.getsize('/kaggle/working/habitlab_model.gguf') / 1e9
    print(f'\nDone! habitlab_model.gguf — {round(size, 1)} GB')
    print('Go to Output tab on the right → download habitlab_model.gguf')
else:
    print('Conversion failed — check errors above')